# Book Club analysis

Parsing and processing data from my Book Club, then scraping additional data from Goodreads to enrich it (for dashboarding).

Author: Mike Aspinall<br>
Date: 2026-07<br>
Notebook Version: v0.1

<hr>

## 1. Project Overview

<strong>Objective</strong><br>
What question/problem are we trying to solve?

<strong>Success Criteria</strong><br>
How will success be measured?

<strong>Dataset(s)</strong>
- Dataset A
- Dataset B

<strong>Key Hypotheses / Questions</strong><br>
- Hypothesis 1
- Hypothesis 2

<strong>Constraints / Assumptions</strong><br>
- Assumption 1
- Assumption 2

<strong>Notes</strong><br>
Any relevant business or technical context.

<hr>

## 2. Environment & Configuration

In [1]:
# Standard libraries
from pathlib import Path
import warnings
import os
import sys
import re
import random
import time
import requests

# Data manipulation
import numpy as np
import pandas as pd

# Web scraping
from bs4 import BeautifulSoup

# Notebook display
from IPython.display import display # for displaying dataframes in a more readable format

# Config
warnings.filterwarnings("ignore")

# Pandas display options
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

# Reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print("✅ Imported libraries!")

✅ Imported libraries!


In [2]:
# Paths - avoids fragile hardcoded file paths

# Create PROJECT_ROOT from the parent of the current file's directory
PROJECT_ROOT = Path.cwd().parent
import sys
sys.path.append(str(PROJECT_ROOT))

DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"	# Creates /my_project/data/raw
DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"	# Creates /my_project/data/processed

FUNCTIONS_UTILS_DIR = PROJECT_ROOT / "functions" / "utils.py"	# Creates /my_project/functions/utils
FUNCTIONS_PREPROCESSING_DIR = PROJECT_ROOT / "functions" / "preprocessing.py"	# Creates /my_project/functions/preprocessing

OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)	# Creates the outputs folder if doesn't exist

print("✅ Created file paths!")

✅ Created file paths!


In [3]:
from functions import preprocessing # functions to extract, clean, and preprocess data
from functions import utils         # functions to visualise, model, or analyse data

print("✅ Imported functions! Access them with: utils.function_name() or preprocessing.function_name()")

✅ Imported functions! Access them with: utils.function_name() or preprocessing.function_name()


In [4]:
import platform

print("Python:", sys.version)
print("Platform:", platform.platform())
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)

Python: 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
Platform: Windows-10-10.0.26200-SP0
Pandas: 2.0.3
NumPy: 1.26.4


In [5]:
# OPTIONAL: Save current environment in a requirements.txt file for future reference
# pip freeze > requirements.txt

<hr>

## 3. Data Loading

<strong>Book Club reviews</strong>
- <strong>source</strong>: Manually compiled spreadsheet
- <strong>extraction method</strong>: csv import
- <strong>filters (date range etc)</strong>: All time / no filters
- <strong>expected grain (what each row represents)</strong>: n/a

In [6]:
book_club_raw = pd.read_csv(DATA_RAW_DIR / "book_club.csv")

print(book_club_raw.shape)
display(book_club_raw.head(1))

(998, 26)


,Unnamed: 0,The Night Circus (Clare),Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24,Unnamed: 25
0,NaN,Mike,Nia,Kath,Jen,Dunc,Gareth,Clare,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


<strong>Goodreads links</strong>
- <strong>source</strong>: Manually compiled spreadsheet
- <strong>extraction method</strong>: csv import
- <strong>filters (date range etc)</strong>: All time / no filters
- <strong>expected grain (what each row represents)</strong>: 1 book

In [7]:
goodreads_links = pd.read_csv(DATA_RAW_DIR / "goodreads_links.csv")

print(goodreads_links.shape)
display(goodreads_links.head(1))

(34, 2)


,Book,link
0,The Night Circus,https://www.goodreads.com/book/show/9361589-th...


<hr>

## 4. Data Preprocessing and Quality Audit

The raw book club data is not in a well-structured tabular format; it needs to be processed into a consistently structured dataframe with logical headers and row data. A function has been developed to do this.

In [8]:
ratings = preprocessing.parse_stacked_ratings(book_club_raw)

In [9]:
# Drop rows with missing score
ratings = ratings[(~ratings["Score"].isna()) &
                  (ratings["Score"] != None) & 
                  (ratings["Score"] != "None") &
                  (ratings["Score"] != "#DIV/0!") &
                  (ratings["Score"] != "0") &
                  (ratings["Score"] != 0)
                  ]
ratings['Score'] = pd.to_numeric(ratings['Score'], downcast='float')

# Tidy up names (Dunc/Duncan, Fred/Freddie, etc)
names_dict = {'Fred': 'Freddie',
              'Dunc': 'Duncan',
              'Katherine': 'Kath',
              'Jennifer': 'Jen'}

ratings['Chooser'] = ratings['Chooser'].replace(names_dict)
ratings['Reviewer'] = ratings['Reviewer'].replace(names_dict)

Next, we can merge in the Goodreads links.

In [10]:
ratings = ratings.merge(goodreads_links, on='Book', how='left')

ratings.info()
ratings.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1090 entries, 0 to 1089
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Book       1090 non-null   object 
 1   Chooser    1090 non-null   object 
 2   Criterion  1090 non-null   object 
 3   Reviewer   1090 non-null   object 
 4   Score      1090 non-null   float32
 5   link       1090 non-null   object 
dtypes: float32(1), object(5)
memory usage: 47.0+ KB


,Book,Chooser,Criterion,Reviewer,Score,link
0,Ask The Dust,Duncan,Plot,Mike,2.0,https://www.goodreads.com/book/show/46227.Ask_...
1,Ask The Dust,Duncan,Plot,Nia,2.0,https://www.goodreads.com/book/show/46227.Ask_...
2,Ask The Dust,Duncan,Plot,Kath,3.0,https://www.goodreads.com/book/show/46227.Ask_...
3,Ask The Dust,Duncan,Plot,Jen,5.0,https://www.goodreads.com/book/show/46227.Ask_...
4,Ask The Dust,Duncan,Plot,Duncan,4.0,https://www.goodreads.com/book/show/46227.Ask_...


<hr>

## 5. EDA

In [11]:
# Avg of reviews of someone's own book choice
self_review_avg = ratings['Score'][(ratings['Criterion'] == 'Total') &
                         (ratings['Chooser'] == ratings['Reviewer'])
                         ].mean()

# Avg reviews overall
review_avg = ratings['Score'][(ratings['Criterion'] == 'Total')].mean()


print(f"Avg total review score when someone reviews their own choice: {self_review_avg:.2f}")
print(f"Avg total review score overall: {review_avg:.2f}")

print(f"-> People reviewed their own books ~{(((self_review_avg/review_avg)*100)-100):.2f}% ({self_review_avg-review_avg:.2f}ppt) more favourably")

Avg total review score when someone reviews their own choice: 7.47
Avg total review score overall: 6.79
-> People reviewed their own books ~9.97% (0.68ppt) more favourably


<hr>

## 6. Web scrape data enrichment

We will now scrape Goodreads for additional information about each of the books, and use that to enrich the data.

In [12]:
# Create dictionary of books and URLs to scrape
books_urls = ratings[['Book', 'link']].drop_duplicates()
# Convert to dictionary
book_dict = dict(books_urls.values)
book_dict

{'Ask The Dust': 'https://www.goodreads.com/book/show/46227.Ask_the_Dust',
 'I Know Why The Caged Bird Sings': 'https://www.goodreads.com/book/show/13214.I_Know_Why_the_Caged_Bird_Sings',
 'Circe': 'https://www.goodreads.com/book/show/35959740-circe',
 'The Beekeeper of Aleppo': 'https://www.goodreads.com/book/show/43124137-the-beekeeper-of-aleppo',
 "All That's Left In The World": 'https://www.goodreads.com/book/show/58329296-all-that-s-left-in-the-world',
 'Apeirogon': 'https://www.goodreads.com/book/show/50024187-apeirogon',
 'The Offing': 'https://www.goodreads.com/book/show/43412959-the-offing',
 'Happiness': 'https://www.goodreads.com/book/show/35458040-happiness',
 'The Wonderful Story of Henry Sugar and Six More': 'https://www.goodreads.com/book/show/6671.The_Wonderful_Story_of_Henry_Sugar_and_Six_More',
 'Crying in H Mart': 'https://www.goodreads.com/book/show/54814676-crying-in-h-mart',
 'A Passage To India': 'https://www.goodreads.com/book/show/45195.A_Passage_to_India',
 'T

In [ ]:
goodreads_data = pd.DataFrame()

for key, value in book_dict.items():
    df = utils.scrape_details(key,value)
    # Combine all books into one master df
    goodreads_data = pd.concat([goodreads_data, df])
            
    # Add a random delay to mimic human behavior
    delay = random.uniform(5, 10)
    time.sleep(delay)

✅ Ask The Dust details scraped
✅ I Know Why The Caged Bird Sings details scraped
✅ Circe details scraped
✅ The Beekeeper of Aleppo details scraped
✅ All That's Left In The World details scraped
✅ Apeirogon details scraped
✅ The Offing details scraped
‼️ Skipping URL 'https://www.goodreads.com/book/show/35458040-happiness': 503 Server Error: Service Unavailable for url: https://www.goodreads.com/book/show/35458040-happiness
❌ Happiness not scraped
✅ The Wonderful Story of Henry Sugar and Six More details scraped
✅ Crying in H Mart details scraped
✅ A Passage To India details scraped
✅ The Dice Man details scraped
✅ Great Circle details scraped
✅ A Visit From The Goon Squad details scraped
✅ The Battle of Ink and Ice details scraped
‼️ Error in publication_date for {book}
✅ The Nicander Chronicles details scraped
✅ Conclave details scraped
✅ Around the World in 80 Trains details scraped
✅ Minna Needs Rehearsal Space details scraped
✅ North Woods details scraped
✅ This Is How You Lose The

In [21]:
goodreads_data.info()
goodreads_data.head(1)

<class 'pandas.core.frame.DataFrame'>
Index: 33 entries, 0 to 0
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Book              33 non-null     object
 1   image             33 non-null     object
 2   author_link       33 non-null     object
 3   author_name       33 non-null     object
 4   genres            32 non-null     object
 5   description       33 non-null     object
 6   goodreads_rating  33 non-null     object
 7   no_ratings        33 non-null     object
 8   no_reviews        33 non-null     object
 9   publication_date  32 non-null     object
dtypes: object(10)
memory usage: 2.8+ KB


,Book,image,author_link,author_name,genres,description,goodreads_rating,no_ratings,no_reviews,publication_date
0,Ask The Dust,https://m.media-amazon.com/images/S/compressed...,https://www.goodreads.com/author/show/25864.Jo...,John Fante,"[Fiction, Classics, American, Literature, Roma...","""Fante was my god."" —Charles Bukowski, in his ...",4.10,40539,3160,"January 1, 1939"


The data has now been scraped, so we can append it to the original ratings.

In [ ]:
master_data = ratings.merge(goodreads_data, on='Book', how='left')
master_data.info()
master_data.head(1)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1090 entries, 0 to 1089
Data columns (total 15 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Book              1090 non-null   object 
 1   Chooser           1090 non-null   object 
 2   Criterion         1090 non-null   object 
 3   Reviewer          1090 non-null   object 
 4   Score             1090 non-null   float32
 5   link              1090 non-null   object 
 6   image             1090 non-null   object 
 7   author_link       1090 non-null   object 
 8   author_name       1090 non-null   object 
 9   genres            1050 non-null   object 
 10  description       1090 non-null   object 
 11  goodreads_rating  1090 non-null   object 
 12  no_ratings        1090 non-null   object 
 13  no_reviews        1090 non-null   object 
 14  publication_date  1050 non-null   object 
dtypes: float32(1), object(14)
memory usage: 123.6+ KB


,Book,Chooser,Criterion,Reviewer,Score,link,image,author_link,author_name,genres,description,goodreads_rating,no_ratings,no_reviews,publication_date
0,Ask The Dust,Duncan,Plot,Mike,2.0,https://www.goodreads.com/book/show/46227.Ask_...,https://m.media-amazon.com/images/S/compressed...,https://www.goodreads.com/author/show/25864.Jo...,John Fante,"[Fiction, Classics, American, Literature, Roma...","""Fante was my god."" —Charles Bukowski, in his ...",4.10,40539,3160,"January 1, 1939"


Finally, lets save the processed dataframes

In [25]:
ratings.to_csv(DATA_PROCESSED_DIR / 'ratings.csv')
goodreads_data.to_csv(DATA_PROCESSED_DIR / 'goodreads_data.csv')
master_data.to_csv(DATA_PROCESSED_DIR / 'master_data.csv')